In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, FloatSlider, HBox, Layout, VBox, HTML, GridBox
from IPython.display import display

# ============================================================
# RANDOM DATA
# ============================================================

np.random.seed(73)

N_max = 2000
D_max = 30

x_source = np.random.randn(N_max + D_max)
v_base = np.random.randn(N_max)

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_cross_spectrum(N=800, delay=10, gain=0.8, noise_std=0.25):

    # --------------------------------------------------------
    # CREATE JOINTLY WSS PROCESSES
    #
    # y[n] = gain * x[n-delay] + v[n]
    # --------------------------------------------------------

    offset = D_max

    x = x_source[offset:offset + N].copy()

    delayed_x = x_source[offset - delay:offset - delay + N].copy()

    y = gain * delayed_x + noise_std * v_base[:N]

    # --------------------------------------------------------
    # REMOVE FINITE-SAMPLE MEANS
    # --------------------------------------------------------

    x = x - np.mean(x)

    y = y - np.mean(y)

    # --------------------------------------------------------
    # CROSS-CORRELATION
    # --------------------------------------------------------

    max_lag = 50

    lags = np.arange(-max_lag, max_lag + 1)

    Rxy = np.zeros(len(lags))

    for i, k in enumerate(lags):

        if k >= 0:

            Rxy[i] = np.mean(
                x[:N-k] *
                y[k:]
            )

        else:

            Rxy[i] = np.mean(
                x[-k:] *
                y[:N+k]
            )

    # --------------------------------------------------------
    # CROSS-SPECTRAL ESTIMATE
    # WELCH-STYLE AVERAGING
    # --------------------------------------------------------

    n_fft = 2048

    segment_length = min(256, N)

    hop = max(segment_length // 2, 1)

    window = np.hanning(segment_length)

    window_power = np.sum(window ** 2)

    segment_starts = list(
        range(
            0,
            N - segment_length + 1,
            hop
        )
    )

    if len(segment_starts) == 0:

        segment_starts = [0]

    Sxy_average = np.zeros(
        n_fft,
        dtype=complex
    )

    for start in segment_starts:

        x_seg = x[start:start + segment_length]

        y_seg = y[start:start + segment_length]

        x_seg = x_seg - np.mean(x_seg)

        y_seg = y_seg - np.mean(y_seg)

        xw = x_seg * window

        yw = y_seg * window

        X_fft = np.fft.fft(
            xw,
            n=n_fft
        )

        Y_fft = np.fft.fft(
            yw,
            n=n_fft
        )

        Sxy_segment = (
            np.conj(X_fft) *
            Y_fft
            /
            window_power
        )

        Sxy_average += Sxy_segment

    Sxy_average = (
        Sxy_average
        /
        len(segment_starts)
    )

    Sxy_average = np.fft.fftshift(
        Sxy_average
    )

    omega = np.linspace(
        -np.pi,
        np.pi,
        n_fft,
        endpoint=False
    )

    # --------------------------------------------------------
    # ESTIMATED MAGNITUDE AND PHASE
    # --------------------------------------------------------

    Sxy_magnitude = np.abs(
        Sxy_average
    )

    Sxy_phase = np.angle(
        Sxy_average
    )

    # --------------------------------------------------------
    # THEORETICAL CROSS-PSD
    # --------------------------------------------------------

    theoretical_magnitude = (
        gain *
        np.ones_like(omega)
    )

    theoretical_phase = (
        -omega *
        delay
    )

    theoretical_phase_wrapped = np.angle(
        np.exp(
            1j *
            theoretical_phase
        )
    )

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(8.0, 9.2)
    )

    # --------------------------------------------------------
    # EXACT AXIS POSITIONS
    #
    # Large vertical gaps are deliberately left between
    # axes for x-labels and legends.
    # --------------------------------------------------------

    ax1 = fig.add_axes(
        [0.11, 0.76, 0.68, 0.17]
    )

    ax2 = fig.add_axes(
        [0.11, 0.43, 0.68, 0.17]
    )

    ax3 = fig.add_axes(
        [0.11, 0.10, 0.68, 0.17]
    )

    # ========================================================
    # GRAPH 1:
    # CROSS-CORRELATION
    # ========================================================

    ax1.plot(
        lags,
        Rxy,
        linewidth=2,
        label='Estimated cross-correlation'
    )

    ax1.axhline(
        0,
        color='k',
        linewidth=0.8
    )

    ax1.axvline(
        0,
        color='k',
        linestyle=':',
        linewidth=0.8
    )

    ax1.axvline(
        delay,
        color='k',
        linestyle='--',
        linewidth=1.3,
        label=f'True delay D = {delay}'
    )

    ax1.set_xlim(
        -max_lag,
        max_lag
    )

    y_limit = max(
        1.15,
        1.25 * np.max(np.abs(Rxy))
    )

    ax1.set_ylim(
        -0.35 * y_limit,
        y_limit
    )

    ax1.set_xlabel(
        'Lag k',
        fontsize=11
    )

    ax1.set_ylabel(
        'Rₓᵧ[k]',
        fontsize=11
    )

    ax1.set_title(
        'Cross-Correlation Between x[n] and y[n]',
        fontsize=12,
        pad=7
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # --------------------------------------------------------
    # LEGEND BELOW GRAPH 1
    # --------------------------------------------------------

    ax1.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.30),
        ncol=2,
        fontsize=8,
        borderaxespad=0.0
    )

    # ========================================================
    # GRAPH 2:
    # CROSS-PSD MAGNITUDE
    # ========================================================

    ax2.plot(
        omega,
        Sxy_magnitude,
        linewidth=1.8,
        label='Estimated |Sₓᵧ(ω)|'
    )

    ax2.plot(
        omega,
        theoretical_magnitude,
        linestyle='--',
        linewidth=1.5,
        label='Theoretical magnitude'
    )

    ax2.set_xlim(
        -np.pi,
        np.pi
    )

    magnitude_limit = max(
        1.25,
        1.30 * np.max(Sxy_magnitude),
        1.30 * gain
    )

    ax2.set_ylim(
        0,
        magnitude_limit
    )

    ax2.set_xticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax2.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax2.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax2.set_ylabel(
        '|Sₓᵧ(ω)|',
        fontsize=11
    )

    ax2.set_title(
        'Magnitude of the Cross-Power Spectral Density',
        fontsize=12,
        pad=7
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # --------------------------------------------------------
    # LEGEND BELOW GRAPH 2
    # --------------------------------------------------------

    ax2.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.30),
        ncol=2,
        fontsize=8,
        borderaxespad=0.0
    )

    # ========================================================
    # GRAPH 3:
    # CROSS-PSD PHASE
    # ========================================================

    ax3.plot(
        omega,
        Sxy_phase,
        linewidth=1.5,
        label='Estimated phase'
    )

    ax3.plot(
        omega,
        theoretical_phase_wrapped,
        linestyle='--',
        linewidth=1.5,
        label='Theoretical phase'
    )

    ax3.set_xlim(
        -np.pi,
        np.pi
    )

    ax3.set_ylim(
        -np.pi - 0.25,
        np.pi + 0.25
    )

    ax3.set_xticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax3.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax3.set_yticks(
        [
            -np.pi,
            -np.pi / 2,
            0,
            np.pi / 2,
            np.pi
        ]
    )

    ax3.set_yticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax3.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax3.set_ylabel(
        'Phase of Sₓᵧ(ω)',
        fontsize=11
    )

    ax3.set_title(
        'Phase of the Cross-Power Spectral Density',
        fontsize=12,
        pad=7
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.6
    )

    # --------------------------------------------------------
    # LEGEND RIGHT OF GRAPH 3
    # VERTICAL
    # --------------------------------------------------------

    ax3.legend(
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        ncol=1,
        fontsize=8,
        borderaxespad=0.0
    )

    plt.show()

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='100px'
)

N_slider = IntSlider(
    min=200,
    max=2000,
    step=100,
    value=800,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

delay_slider = IntSlider(
    min=0,
    max=30,
    step=1,
    value=10,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

gain_slider = FloatSlider(
    min=0.2,
    max=1.5,
    step=0.05,
    value=0.8,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

noise_slider = FloatSlider(
    min=0.0,
    max=1.0,
    step=0.05,
    value=0.25,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# MAXIMUM VALUES
# ============================================================

N_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">2000</div>'
)

delay_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">30</div>'
)

gain_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">1.5</div>'
)

noise_max_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">1.0</div>'
)

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_cross_spectrum,
    N=N_slider,
    delay=delay_slider,
    gain=gain_slider,
    noise_std=noise_slider
)

# ============================================================
# DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 16px;
    line-height: 1.30;
    width: 1050px;
">

<div style="
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 6px;
">
Cross-Correlation and Cross-Power Spectral Density
</div>

<div style="margin-bottom:5px;">
<b>Jointly WSS processes:</b> two processes are jointly WSS when their individual second-order statistics are stationary and their cross-correlation depends only on the time difference.
</div>

<div style="margin-bottom:5px;">
<b>Cross-correlation:</b> Rₓᵧ[k] measures the statistical similarity between x[n] and a shifted version of y[n].
</div>

<div style="margin-bottom:5px;">
<b>Cross-power spectral density:</b> Sₓᵧ(ω) is the Fourier transform of Rₓᵧ[k] and is generally complex-valued.
</div>

<div style="margin-bottom:5px;">
<b>Delay:</b> a time displacement D produces a peak in the cross-correlation and a linear phase term in the cross-spectrum.
</div>

<div>
<b>This notebook:</b> demonstrates how the same statistical relationship appears as a displacement in the lag domain and as phase information in the frequency domain.
</div>

</div>
""")

# ============================================================
# THEORY BLOCK
# ============================================================

theory_block = VBox(
    [
        theory_html
    ],
    layout=Layout(
        margin='0px 0px 18px 0px'
    )
)

# ============================================================
# SLIDER LABELS
# ============================================================

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Samples N:</div>'
)

delay_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Delay D:</div>'
)

gain_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Gain a:</div>'
)

noise_label = HTML(
    '<div style="font-family:Arial; font-size:14px;">Noise std:</div>'
)

# ============================================================
# SLIDER GRID
# ============================================================

slider_grid = GridBox(
    children=[
        N_label, N_slider, N_max_label,
        delay_label, delay_slider, delay_max_label,
        gain_label, gain_slider, gain_max_label,
        noise_label, noise_slider, noise_max_label
    ],
    layout=Layout(
        width='270px',
        grid_template_columns='110px 100px 45px',
        grid_template_rows='30px 30px 30px 30px',
        grid_gap='2px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS
#
# IMPORTANT:
# margin-top places the sliders specifically beside graph 2.
# ============================================================

controls = VBox(
    [
        slider_grid
    ],
    layout=Layout(
        width='280px',
        min_width='280px',
        align_items='flex-start',
        margin='315px 0px 0px -35px',
        overflow='hidden'
    )
)

# ============================================================
# GRAPH OUTPUT + CONTROLS
# ============================================================

graph_and_controls = HBox(
    [
        widget_plot.children[-1],
        controls
    ],
    layout=Layout(
        width='1000px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation_html = HTML("""
<div style="
    font-family: Arial, sans-serif;
    font-size: 15px;
    line-height: 1.35;
    width: 1050px;
    margin-top: 12px;
">

<div style="
    font-size: 18px;
    font-weight: bold;
    margin-bottom: 6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:5px;">
<b>First graph:</b> the cross-correlation has its dominant peak near k = D, revealing the relative delay between the two processes.
</div>

<div style="margin-bottom:5px;">
<b>Second graph:</b> because x[n] is white, the theoretical magnitude of the cross-spectrum is approximately constant and equal to the gain a.
</div>

<div style="margin-bottom:5px;">
<b>Third graph:</b> the delay appears as the phase factor exp(-jωD). The estimated phase is compared directly with the theoretical wrapped linear phase -ωD.
</div>

<div style="
    margin-top:8px;
    margin-bottom:5px;
    font-size:17px;
    font-weight:bold;
">
Peak at lag D &nbsp;&nbsp; ⇔ &nbsp;&nbsp; Linear spectral phase -ωD
</div>

<div>
For D = 0 the phase is zero. For D &gt; 0 the wrapped linear phase produces the characteristic descending segments with jumps between +π and -π.
</div>

</div>
""")

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_block,
        graph_and_controls,
        interpretation_html
    ],
    layout=Layout(
        width='1100px',
        overflow='hidden'
    )
)

display(main_layout)